### Data Fetching

In [ ]:
import psycopg2
import pandas as pd

In [ ]:
def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    try:
        connection = psycopg2.connect(
            host=host_ip, database=database_name, user=user, password=password, port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    finally:
        if 'connection' in locals():
            connection.close()

# --- Configuration ---
HOST_IP = "100.94.14.115"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

In [ ]:
keywords = ["PAPIS"]

pattern = '|'.join(keywords)

df = df_original.copy(deep=True)
df = df[
    (df['status_id'] == 1) &
    (df['dept_name'] == 'Signalling-And-Communication') &
    (df['filename'].str.contains(pattern, case=False, na=False))
][['filename', 'workorder_id', 'json_data']]

df

In [ ]:
import pandas as pd

def rename_keys(d):
    if not isinstance(d, dict):
        return d
    
    mapping = {
        'otn': 'papis',
        'pa_pis_system': 'papis'
    }

    return {mapping.get(k, k): v for k, v in d.items()}

df['json_data'] = df['json_data'].apply(rename_keys)

valid_json = df['json_data'][df['json_data'].apply(lambda x: isinstance(x, dict))]
all_keys = set()
for item in valid_json:
    all_keys.update(item.keys())

print(sorted(all_keys))


In [ ]:
import pandas as pd
import json

df_papis = df.copy(deep=True)

valid_mask = df_papis['json_data'].apply(lambda x: isinstance(x, dict))

rows = []

for _, row in df_papis[valid_mask].iterrows():
    base_data = {
        'workorder_id': row['workorder_id'],
        'filename': row['filename']
    }

    cctv_data = row['json_data'].get('papis', {})

    for key, value in cctv_data.items():
        if isinstance(value, dict):
            base_data[key] = json.dumps(value)
        else:
            base_data[key] = value

    rows.append(base_data)

df_papis = pd.DataFrame(rows)

df_papis = df_papis.drop(
    columns=["pm_order_no", "reference_document_no", "reference_document"],
    errors="ignore"
)

df_papis

In [ ]:
df_papis.columns

In [ ]:
def flatten_procedures(proc_data):
    flat_data = {}
    
    if isinstance(proc_data, str):
        try:
            proc_data = json.loads(proc_data)
        except:
            return {}
            
    if not isinstance(proc_data, dict): 
        return flat_data

    def walk(d, parent_key=''):
        for k, v in d.items():
            new_key = f"{parent_key}.{k}" if parent_key else k
            
            if isinstance(v, dict):
                if 'status' in v:
                    flat_data[f"{new_key}.status"] = v.get('status')
                    flat_data[f"{new_key}.remarks"] = v.get('remarks')
                else:
                    walk(v, new_key)
            else:
                flat_data[new_key] = v

    walk(proc_data)
    return flat_data

def parse_json(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except:
            return {}
    return x

df_papis["procedures"] = df_papis["procedures"].apply(parse_json)
df_papis["pa_zone_status"] = df_papis["pa_zone_status"].apply(parse_json)

df_proc_flat = pd.DataFrame(
    df_papis["procedures"].apply(flatten_procedures).tolist(), 
    index=df_papis.index
)

df_zone_flat = pd.DataFrame(
    df_papis["pa_zone_status"].apply(flatten_procedures).tolist(), 
    index=df_papis.index
).add_prefix("pa_zone_status.")

df_final = pd.concat([
    df_papis.drop(columns=['procedures', 'pa_zone_status']), 
    df_proc_flat, 
    df_zone_flat
], axis=1)

print(df_final.columns)

In [ ]:
list(df_final.columns)

In [ ]:
import numpy as np

pd.set_option('future.no_silent_downcasting', True)

status_cols = df_final.filter(like='.status').columns
df_final[status_cols] = df_final[status_cols].apply(lambda x: x.str.lower().replace('na', 'N/A') if x.dtype == 'object' else x)

remarks_cols = df_final.filter(like='.remarks').columns
df_final[remarks_cols] = df_final[remarks_cols].apply(lambda x: x.str.lower().replace('na', np.nan) if x.dtype == 'object' else x)

In [ ]:
output_file = f"../../output/snc/papis.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_final.to_excel(writer, sheet_name='papis', index=False)

print(f"Saved excel to {output_file}")